# 🎯 Pricer Fine-Tuning V2 - Professional Pipeline

This improved version implements best-practice ML workflows:
- **Train/Val Split**: Automated 90/10 split to monitor overfitting.
- **Dataset Balancing**: (Conceptual) logic to ensure price distribution coverage.
- **Job Monitor**: Real-time polling of fine-tuning status and even logging.
- **HuggingFace Ready**: Formatting compatible with `datasets` library.

In [ ]:
import os
import json
import random
import time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
client = OpenAI()

## 1. Advanced Data Preparation

We split our data into training and validation sets.

In [ ]:
def format_message(item):
    return {
        "messages": [
            {"role": "system", "content": "You are a pricing expert. Estimate the retail price of the product based on its description."},
            {"role": "user", "content": item['description']},
            {"role": "assistant", "content": f"${item['price']:.2f}"}
        ]
    }

def prepare_datasets(raw_data, train_file="train.jsonl", val_file="val.jsonl", split_ratio=0.9):
    random.shuffle(raw_data)
    split_idx = int(len(raw_data) * split_ratio)
    
    train_data = raw_data[:split_idx]
    val_data = raw_data[split_idx:]
    
    with open(train_file, 'w') as f:
        for item in train_data: f.write(json.dumps(format_message(item)) + '\n')
        
    with open(val_file, 'w') as f:
        for item in val_data: f.write(json.dumps(format_message(item)) + '\n')
            
    print(f"Created {train_file} ({len(train_data)} items) and {val_file} ({len(val_data)} items)")

## 2. Fine-Tuning Execution & Monitoring

Handling file uploads and polling for status.

In [ ]:
def run_fine_tune(train_path, val_path):
    # 1. Upload
    t_file = client.files.create(file=open(train_path, "rb"), purpose="fine-tune")
    v_file = client.files.create(file=open(val_path, "rb"), purpose="fine-tune")
    
    # 2. Start Job
    job = client.fine_tuning.jobs.create(
        training_file=t_file.id,
        validation_file=v_file.id,
        model="gpt-4.1-nano-2025-04-14",
        suffix="v2_pricer"
    )
    return job.id

def monitor_job(job_id):
    print(f"Monitoring Job: {job_id}")
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        status = job.status
        print(f"[{time.strftime('%H:%M:%S')}] Status: {status}")
        if status in ['succeeded', 'failed', 'cancelled']:
            break
        time.sleep(30)

## 3. Mock Run

Testing the preparation logic.

In [ ]:
mock_data = [
    {"description": "Gaming Mouse with RGB", "price": 45.99},
    {"description": "Mechanical Keyboard", "price": 120.00},
    {"description": "HDMI Cable 2m", "price": 15.00},
    {"description": "Laptop Cooling Pad", "price": 25.50},
    {"description": "USB-C Hub 7-in-1", "price": 39.99}
]

prepare_datasets(mock_data)